# 02 – Huấn luyện M4 (Logistic Regression học trọng số)

Ghép hai file nhãn độc lập, tính kappa, chốt nhãn theo quy tắc (lệch >= 2 bậc: thảo luận; lệch 1 bậc: trung bình làm tròn xuống), nhãn nhị phân "phù hợp" khi nhãn >= 2, chia **GroupKFold 5 theo `candidate_id`** để không rò rỉ.

In [1]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "backend"))

# Có thể đổi bằng biến môi trường khi chạy thử trên dữ liệu khác
DATA = Path(os.getenv("TH3_DATA_DIR", ROOT / "backend/seed/data"))
LABELS = Path(os.getenv("TH3_LABELS_DIR", ROOT / "backend/seed/labels"))
OUTDIR = Path(os.getenv("TH3_OUT_DIR", ROOT / "docs/report/tables"))
MODELS_OUT = Path(os.getenv("TH3_MODELS_OUT", ROOT / "backend/models_store"))
OUTDIR.mkdir(parents=True, exist_ok=True)


In [2]:
import numpy as np
import pandas as pd
from app.ai import datasets, metrics
from app.ai.embedder import get_embedder

emb = get_embedder()
print("Embedder:", emb.name)  # st:... là model thật; hash-384 chỉ để chạy thử, KHÔNG dùng số liệu để báo cáo
jobs = datasets.load_jobs(DATA / "jobs.csv")
cands = datasets.load_candidates(DATA / "candidates_generated.csv")
datasets.attach_embeddings(jobs, emb)
datasets.attach_embeddings(cands, emb)

LABEL_A = os.getenv("TH3_LABELS_A", "labels_pairs_hung.csv")
LABEL_B = os.getenv("TH3_LABELS_B", "labels_pairs_hoanganh.csv")
raw = datasets.load_labels(LABELS / LABEL_A, LABELS / LABEL_B)
col_a, col_b = [c for c in raw.columns if c.startswith("label_")]
print(f"{len(raw)} cặp có đủ nhãn của cả hai người")


/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Could not reach the Hub (Cannot reach https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2046.06it/s]

Embedder: st:paraphrase-multilingual-MiniLM-L12-v2


300 cặp có đủ nhãn của cả hai người


In [3]:
kappa = metrics.weighted_kappa(raw[col_a].to_numpy(), raw[col_b].to_numpy())
print(f"Cohen's kappa (trọng số bình phương) = {kappa:.3f}  (mục tiêu >= 0,6)")

raw["label"] = metrics.merge_labels(raw[col_a].to_numpy(), raw[col_b].to_numpy())
disputed = raw[raw["label"] < 0]
print(f"{len(disputed)} cặp lệch >= 2 bậc cần hai người thảo luận và chốt lại")
disputed.to_csv(OUTDIR / "labels_disputed.csv", index=False)
labels = raw[raw["label"] >= 0].reset_index(drop=True)  # SAU KHI thảo luận: cập nhật file nhãn rồi chạy lại ô này
labels[["candidate_id", "job_id", "label"]].to_csv(OUTDIR / "labels_pairs_final.csv", index=False)
pd.DataFrame([{"kappa_quadratic": kappa, "pairs": len(raw), "disputed": len(disputed)}]).to_csv(OUTDIR / "kappa.csv", index=False)


Cohen's kappa (trọng số bình phương) = 0.959  (mục tiêu >= 0,6)
0 cặp lệch >= 2 bậc cần hai người thảo luận và chốt lại


In [4]:
from app.ai import evaluate, train
from sklearn.metrics import roc_auc_score

X, feats = evaluate.build_feature_matrix(labels, cands, jobs, emb)
y = (labels["label"] >= 2).astype(int).to_numpy()
groups = labels["candidate_id"].to_numpy()
print("Số cặp:", len(y), "| tỉ lệ phù hợp:", round(y.mean(), 3), "| số ứng viên:", len(set(groups)))

# Ba họ mô hình học máy, cùng 6 đặc trưng, cùng GroupKFold theo ứng viên (không ứng viên nào vừa huấn luyện vừa kiểm tra).
#  M4 Logistic Regression: C được chọn BÊN TRONG mỗi fold huấn luyện (GroupKFold lồng) nên điểm kiểm tra không rò rỉ.
#  M5 Random Forest, M6 Gradient Boosting: siêu tham số cố định từ trước (nông, regularization), không dò trên dữ liệu kiểm tra.
oof = {
    "m4": train.cross_val_scores(X, y, groups, n_splits=5, kind="logreg", tune=True),
    "m5": train.cross_val_scores(X, y, groups, n_splits=5, kind="random_forest"),
    "m6": train.cross_val_scores(X, y, groups, n_splits=5, kind="gradient_boosting"),
}
oof_untuned = train.cross_val_scores(X, y, groups, n_splits=5, kind="logreg")
auc = {k: roc_auc_score(y, v) for k, v in oof.items()}
auc["m4 (C=1, chưa chọn tham số)"] = roc_auc_score(y, oof_untuned)
for k, v in auc.items():
    print(f"ROC-AUC trên fold kiểm tra (GroupKFold) – {k}: {v:.4f}")

oof_table = pd.DataFrame({"candidate_id": labels["candidate_id"], "job_id": labels["job_id"],
                          **{f"{k}_oof_score": v * 100 for k, v in oof.items()}})
oof_table.to_csv(OUTDIR / "ml_oof.csv", index=False)
oof_table[["candidate_id", "job_id", "m4_oof_score"]].to_csv(OUTDIR / "m4_oof.csv", index=False)  # tương thích bản cũ
pd.DataFrame({"model": list(auc), "roc_auc_oof": list(auc.values())}).to_csv(OUTDIR / "ml_auc.csv", index=False)

Số cặp: 300 | tỉ lệ phù hợp: 0.297 | số ứng viên: 30


ROC-AUC trên fold kiểm tra (GroupKFold) – m4: 0.9768
ROC-AUC trên fold kiểm tra (GroupKFold) – m5: 0.9810
ROC-AUC trên fold kiểm tra (GroupKFold) – m6: 0.9673
ROC-AUC trên fold kiểm tra (GroupKFold) – m4 (C=1, chưa chọn tham số): 0.9809


In [5]:
best_c = train.best_logreg_c(X, y, groups)
print("C của M4 chọn trên toàn bộ dữ liệu:", best_c, "(lưới", train.LOGREG_C_GRID, ")")
models = {
    "m4": train.fit_model("logreg", X, y, C=best_c),
    "m5": train.fit_model("random_forest", X, y),
    "m6": train.fit_model("gradient_boosting", X, y),
}
model = models["m4"]
weights = pd.DataFrame({
    "feature": list(train.FEATURE_NAMES),
    "m4_coef": model.coef_[0],
    "m5_importance": models["m5"].feature_importances_,
    "m6_importance": models["m6"].feature_importances_,
    "m3_hand_weight": [train.M3_DEFAULT_WEIGHTS[n] for n in train.FEATURE_NAMES],
})
weights.to_csv(OUTDIR / "m4_coefficients.csv", index=False)  # tên cũ, nay gồm cả độ quan trọng đặc trưng của M5, M6
weights.round(4)

C của M4 chọn trên toàn bộ dữ liệu: 10.0 (lưới (0.1, 0.3, 1.0, 3.0, 10.0) )


,feature,m4_coef,m5_importance,m6_importance,m3_hand_weight
0,skill_req,10.8152,0.6376,0.8019,0.40
1,skill_pref,2.3220,0.0502,0.0118,0.15
2,semantic,4.3407,0.2441,0.0687,0.25
3,exp,2.8055,0.0465,0.1031,0.10
4,edu,2.4096,0.0094,0.0073,0.05
5,pref,-0.5518,0.0121,0.0072,0.05


In [6]:
version = train.save_model(models["m4"], MODELS_OUT / "m4.joblib", "m4", version=os.getenv("TH3_M4_VERSION") or None,
                          extra={"kappa": kappa, "n_pairs": int(len(y)), "embedder": emb.name, "C": best_c})
for mid in ("m5", "m6"):
    v = train.save_model(models[mid], MODELS_OUT / f"{mid}.joblib", mid, version=version.replace("m4", mid, 1),
                         extra={"kappa": kappa, "n_pairs": int(len(y)), "embedder": emb.name})
    print("Đã lưu", MODELS_OUT / f"{mid}.joblib", "| phiên bản", v)
print("Đã lưu", MODELS_OUT / "m4.joblib", "| phiên bản", version)

# Chỉ đặt UPDATE_M3 = True nếu bảng ở notebook 03 cho thấy M4 tốt hơn M3 (hệ số dương chuẩn hoá về tổng 1).
UPDATE_M3 = False
if UPDATE_M3:
    w = train.m3_weights_from_m4(model)
    train.save_m3_weights(w, MODELS_OUT / "m3_weights.json", f"m3-from-{version}")
    print("Đã cập nhật trọng số M3:", w)

Đã lưu /work/backend/models_store/m5.joblib | phiên bản m5-ai-labels
Đã lưu /work/backend/models_store/m6.joblib | phiên bản m6-ai-labels
Đã lưu /work/backend/models_store/m4.joblib | phiên bản m4-ai-labels
